# Message

Message là đơn vị ngữ cảnh cơ bản cho các model trong LangChain. Chúng đại diện cho đầu vào và đầu ra của các model, mang theo cả nội dung và metadata cần thiết để thể hiện trạng thái của một cuộc hội thoại khi tương tác với một LLM.

Message là các đối tượng bao gồm:

* **Role** - Xác định loại message (ví dụ: `system`, `user`)
* **Content** - Thể hiện nội dung thực tế của message (như văn bản, hình ảnh, âm thanh, tài liệu, v.v.)
* **Metadata** - Các trường tùy chọn như thông tin phản hồi, ID của message và lượng token đã sử dụng

LangChain cung cấp một loại message tiêu chuẩn hoạt động xuyên suốt qua tất cả các nhà cung cấp model, đảm bảo tính nhất quán bất kể bạn đang gọi model nào.

## Cách sử dụng cơ bản

Cách đơn giản nhất để sử dụng message là tạo các đối tượng message và truyền chúng vào một model khi [invoke](https://docs.langchain.com/oss/python/langchain/models#invocation).

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage

model = init_chat_model("google_genai:gemini-3.5-flash-lite")

system_msg = SystemMessage("Bạn là một trợ lý hữu ích.")
human_msg = HumanMessage("Xin chào, bạn khỏe không?")

# Sử dụng với các chat model
messages = [system_msg, human_msg]
response = model.invoke(messages)

<div class="alert alert-success">

Các [agent](https://docs.langchain.com/oss/python/langchain/agents) đa lượt sẽ tích lũy lịch sử message rất dài. [LangSmith](https://docs.langchain.com/langsmith/observability) ghi lại từng lượt, kết quả của tool và phản hồi của model để bạn có thể kiểm tra toàn bộ cuộc hội thoại. Hãy làm theo [hướng dẫn bắt đầu nhanh về tracing](https://docs.langchain.com/langsmith/trace-with-langchain) để bật tính năng tracing.

Chúng tôi khuyên bạn cũng nên thiết lập [LangSmith Engine](https://docs.langchain.com/langsmith/engine) để giám sát các trace, phát hiện sự cố và đề xuất cách khắc phục.

</div>

### Text prompt

Text prompt là các string - lý tưởng cho các tác vụ tạo văn bản đơn giản, nơi bạn không cần phải lưu giữ lịch sử hội thoại.

In [ ]:
response = model.invoke("Viết một bài thơ haiku về mùa xuân")

**Sử dụng text prompt khi:**

* Bạn có một yêu cầu duy nhất, độc lập
* Bạn không cần lưu trữ lịch sử hội thoại
* Bạn muốn code ít phức tạp nhất

### Message prompt

Ngoài ra, bạn có thể truyền một danh sách các message vào model bằng cách cung cấp một list chứa các đối tượng message.

In [ ]:
messages = [
    SystemMessage("Bạn là một chuyên gia về thơ"),
    HumanMessage("Viết một bài thơ haiku về mùa xuân"),
]
response = model.invoke(messages)

**Sử dụng message prompt khi:**

* Quản lý các cuộc hội thoại nhiều lượt
* Làm việc với nội dung đa phương thức như hình ảnh, âm thanh, tệp
* Cần đưa vào system instruction

### Định dạng dictionary

Bạn cũng có thể định nghĩa message trực tiếp theo định dạng chat completions của OpenAI.

In [ ]:
messages = [
    {"role": "system", "content": "Bạn là một chuyên gia về thơ"},
    {"role": "user", "content": "Viết một bài thơ haiku về mùa xuân"},
    {"role": "assistant", "content": "Hoa anh đào nở rộ..."}
]
response = model.invoke(messages)

## Các loại message

* System message - Chỉ đạo model cách hoạt động và cung cấp ngữ cảnh cho các tương tác
* Human message - Đại diện cho đầu vào của người dùng và các tương tác với model
* AI message - Các phản hồi được model tạo ra, bao gồm nội dung văn bản, tool call và metadata
* Tool message - Đại diện cho đầu ra từ các [tool call](https://docs.langchain.com/oss/python/langchain/models#tool-calling)

### System message

Một [`SystemMessage`](https://reference.langchain.com/python/langchain-core/messages/system/SystemMessage) đại diện cho một tập hợp các chỉ thị ban đầu nhằm định hướng hành vi của model. Bạn có thể sử dụng system message để thiết lập giọng điệu, xác định vai trò của model và thiết lập các nguyên tắc cho câu trả lời.

In [ ]:
# Các chỉ thị cơ bản
system_msg = SystemMessage("Bạn là một trợ lý lập trình hữu ích.")

messages = [
    system_msg,
    HumanMessage("Làm cách nào để tôi tạo một REST API?")
]
response = model.invoke(messages)

In [ ]:
# Persona chi tiết
system_msg = SystemMessage("""
Bạn là một lập trình viên Python senior có chuyên môn về các web framework.
Luôn cung cấp ví dụ code và giải thích lý do của bạn.
Hãy giải thích ngắn gọn nhưng cặn kẽ.
""")

messages = [
    system_msg,
    HumanMessage("Làm cách nào để tôi tạo một REST API?")
]
response = model.invoke(messages)

### Human message

Một [`HumanMessage`](https://reference.langchain.com/python/langchain-core/messages/human/HumanMessage) đại diện cho đầu vào và các tương tác của người dùng. Chúng có thể chứa văn bản, hình ảnh, âm thanh, tệp và bất kỳ lượng nội dung đa phương thức nào khác.

#### Text content

In [ ]:
# Đối tượng message
response = model.invoke([
    HumanMessage("Machine learning là gì?")
])

In [ ]:
# Sử dụng một string là cách viết tắt cho một HumanMessage duy nhất
response = model.invoke("Machine learning là gì?")

#### Metadata của message

In [ ]:
human_msg = HumanMessage(
    content="Xin chào!",
    name="alice",  # Tùy chọn: xác định các người dùng khác nhau
    id="msg_123",  # Tùy chọn: định danh duy nhất dùng cho tracing
)

<div class="alert alert-info">

Hành vi của trường `name` thay đổi tùy theo provider - một số sử dụng nó để nhận dạng người dùng, số khác lại bỏ qua. Để kiểm tra, hãy tham khảo [tài liệu tham khảo](https://reference.langchain.com/python/integrations/) của các provider.

</div>

### AI message

Một [`AIMessage`](https://reference.langchain.com/python/langchain-core/messages/ai/AIMessage) đại diện cho đầu ra của một lần gọi model. Chúng có thể bao gồm dữ liệu đa phương thức, tool call và các metadata đặc thù của provider mà bạn có thể truy cập sau này.

In [ ]:
response = model.invoke("Giải thích về AI")
print(type(response))

<class 'langchain_core.messages.ai.AIMessage'>


Các đối tượng [`AIMessage`](https://reference.langchain.com/python/langchain-core/messages/ai/AIMessage) được model trả về khi gọi nó, chứa tất cả metadata liên kết trong phản hồi.

Các provider đánh giá/đặt ngữ cảnh cho các loại message theo những cách khác nhau, điều đó có nghĩa là đôi khi việc tạo thủ công một đối tượng [`AIMessage`](https://reference.langchain.com/python/langchain-core/messages/ai/AIMessage) và chèn nó vào lịch sử hội thoại như thể nó đến từ model sẽ rất hữu ích.

In [ ]:
from langchain.messages import AIMessage

# Tạo một AI message thủ công (ví dụ: cho lịch sử hội thoại)
ai_msg = AIMessage("Tôi rất sẵn lòng giúp bạn với câu hỏi đó!")

# Thêm vào lịch sử hội thoại
messages = [
    SystemMessage("Bạn là một trợ lý hữu ích"),
    HumanMessage("Bạn có thể giúp tôi được không?"),
    ai_msg,  # Chèn vào như thể nó đến từ model
    HumanMessage("Tuyệt vời! 2+2 bằng mấy?")
]

response = model.invoke(messages)

**Các thuộc tính:**

- `text` (`string`): Nội dung văn bản của message.
- `content` (`string | dict[]`): Nội dung thô của message.
- `content_blocks` (`ContentBlock[]`): Các content block được chuẩn hóa của message.
- `tool_calls` (`dict[] | None`): Các tool call được thực hiện bởi model. Sẽ trống nếu không có tool nào được gọi.
- `id` (`string`): Một định danh duy nhất cho message (có thể được LangChain tạo tự động hoặc trả về trong phản hồi của provider)
- `usage_metadata` (`dict | None`): Metadata về mức sử dụng của message, có thể chứa số lượng token khi khả dụng.
- `response_metadata` (`ResponseMetadata | None`): Metadata phản hồi của message.

#### Tool call

Khi các model thực hiện [tool call](https://docs.langchain.com/oss/python/langchain/models#tool-calling), chúng sẽ được đưa vào bên trong [`AIMessage`](https://reference.langchain.com/python/langchain-core/messages/ai/AIMessage):

In [3]:
def get_weather(location: str) -> str:
    """Lấy thông tin thời tiết tại một địa điểm."""
    ...

model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke("Thời tiết ở Paris như thế nào?")

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(f"ID: {tool_call['id']}")

Tool: get_weather
Args: {'location': 'Paris'}
ID: call_284984


Các dữ liệu có cấu trúc khác, chẳng hạn như quá trình suy luận hoặc trích dẫn, cũng có thể xuất hiện trong [nội dung](https://docs.langchain.com/oss/python/langchain/messages#message-content) của message.

#### Token usage

Một [`AIMessage`](https://reference.langchain.com/python/langchain-core/messages/ai/AIMessage) có thể lưu giữ số lượng token và các metadata sử dụng khác trong trường [`usage_metadata`](https://reference.langchain.com/python/langchain-core/messages/ai/UsageMetadata) của nó:

In [ ]:
response = model.invoke("Xin chào!")
response.usage_metadata

{'input_tokens': 4,
 'output_tokens': 13,
 'total_tokens': 17,
 'input_token_details': {'cache_read': 0}}

Xem [`UsageMetadata`](https://reference.langchain.com/python/langchain-core/messages/ai/UsageMetadata) để biết thêm chi tiết.

#### Streaming và các chunk

Trong quá trình streaming, bạn sẽ nhận được các đối tượng [`AIMessageChunk`](https://reference.langchain.com/python/langchain-core/messages/ai/AIMessageChunk) có thể kết hợp lại thành một đối tượng message hoàn chỉnh:

In [10]:
chunks = []
full_message = None
for chunk in model.stream("Xin chào"):
    chunks.append(chunk)
    print(chunk.text)
    full_message = chunk if full_message is None else full_message + chunk

full_message

Xin
 chào! Tôi có thể giúp gì cho bạn hôm nay?




AIMessageChunk(content=[{'type': 'text', 'text': 'Xin chào! Tôi có thể giúp gì cho bạn hôm nay?', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPII0DpJs2mH7t2YZa0hKn6icwntAb2UgJ2ph2zQeshbBddHUdcK4W3/Hi8aYoIPO6izYOmDw0sBBwi/4NjzmA29d4GUI6pSTIFtkUIddgtYDT+t6MZ3I3'}}], additional_kwargs={}, response_metadata={'safety_ratings': [], 'model_provider': 'google_genai', 'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite'}, id='lc_run--01a04238-c0f0-7f53-8851-b6d1e21e716d', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 3, 'output_tokens': 13, 'total_tokens': 16, 'input_token_details': {'cache_read': 0}}, tool_call_chunks=[], chunk_position='last')

<div class="alert alert-info">

Tìm hiểu thêm:

* [Streaming token từ các chat model](https://docs.langchain.com/oss/python/langchain/models#stream)
* [Streaming token và/hoặc các bước từ agent](https://docs.langchain.com/oss/python/langchain/streaming)

</div>

### Tool message

Đối với các model hỗ trợ [tool calling](https://docs.langchain.com/oss/python/langchain/models#tool-calling), AI message có thể chứa các tool call. Tool message được sử dụng để truyền kết quả của một lần thực thi tool duy nhất trở lại cho model.

Các [tool](https://docs.langchain.com/oss/python/langchain/tools) có thể trực tiếp tạo ra các đối tượng [`ToolMessage`](https://reference.langchain.com/python/langchain-core/messages/tool/ToolMessage). Dưới đây, chúng tôi hiển thị một ví dụ đơn giản. Đọc thêm trong [hướng dẫn về tool](https://docs.langchain.com/oss/python/langchain/tools).

In [ ]:
from langchain.messages import ToolMessage

# Sau khi model thực hiện một tool call
# (Ở đây, chúng tôi trình bày cách tạo message thủ công cho ngắn gọn)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Thực thi tool và tạo message kết quả
weather_result = "Trời nắng, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Phải khớp với ID của cuộc gọi (call ID)
)

# Tiếp tục cuộc hội thoại
messages = [
    HumanMessage("Thời tiết ở San Francisco như thế nào?"),
    ai_message,  # Tool call của model
    tool_message,  # Kết quả thực thi tool
]
response = model.invoke(messages)  # Model xử lý kết quả

**Các thuộc tính:**

* `content`(`string`, required): Đầu ra dạng chuỗi của tool call.
* `tool_call_id` (`string`, required): ID của tool call mà message này đang phản hồi. Phải khớp với ID của tool call trong [`AIMessage`](https://reference.langchain.com/python/langchain-core/messages/ai/AIMessage).
* `name` (`string`, required): Tên của tool đã được gọi.
* `artifact` (`dict`): Dữ liệu bổ sung không được gửi tới model nhưng có thể được truy cập thông qua code.

Trường `artifact` lưu trữ dữ liệu bổ sung sẽ không được gửi đến model nhưng có thể truy cập được thông qua code. Điều này rất hữu ích để lưu trữ kết quả thô, thông tin debug hoặc dữ liệu để xử lý ở các bước tiếp theo mà không làm lộn xộn ngữ cảnh của model.

Ví dụ, một tool [retrieval](https://docs.langchain.com/oss/python/deepagents/retrieval) có thể lấy một đoạn văn từ tài liệu để model tham chiếu. Trong khi `content` của message chứa văn bản mà model sẽ tham chiếu, thì một `artifact` có thể chứa mã định danh tài liệu hoặc metadata khác mà ứng dụng có thể sử dụng (ví dụ: để render lên một trang web). Xem ví dụ dưới đây:

In [ ]:
# Được gửi tới model
message_content = "Đó là thời kỳ tuyệt vời nhất, cũng là thời kỳ tồi tệ nhất."

# Artifact có sẵn cho downstream
artifact = {"document_id": "doc_123", "page": 0}

tool_message = ToolMessage(
    content=message_content,
    tool_call_id="call_123",
    name="search_books",
    artifact=artifact,
)

Xem [hướng dẫn về RAG](https://docs.langchain.com/oss/python/deepagents/rag) để có ví dụ thực tế về việc xây dựng các [agent](https://docs.langchain.com/oss/python/langchain/agents) retrieval với LangChain.

## Nội dung của message

Bạn có thể nghĩ nội dung của một message giống như payload dữ liệu được gửi đến model. Các message có một thuộc tính `content` được định kiểu lỏng, hỗ trợ cả chuỗi và danh sách các đối tượng không định kiểu (ví dụ: dictionary). Điều này cho phép hỗ trợ trực tiếp các cấu trúc nguyên bản của provider trong các chat model của LangChain, chẳng hạn như nội dung multimodal và các dữ liệu khác.

Tách biệt với đó, LangChain cung cấp các loại nội dung chuyên dụng cho văn bản, quá trình reasoning, trích dẫn, dữ liệu đa phương thức, các tool call ở phía server và các nội dung message khác. Xem các content block tiêu chuẩn bên dưới.

Chat model của LangChain chấp nhận nội dung message trong thuộc tính `content`.

Thuộc tính này có thể chứa một trong số các loại sau:

1. Một chuỗi (string)
2. Một list các content block theo định dạng nguyên bản của nhà cung cấp
3. Một list các content block tiêu chuẩn của LangChain

Xem ví dụ dưới đây sử dụng đầu vào multimodal:

In [ ]:
# Nội dung dạng chuỗi
human_message = HumanMessage("Xin chào, bạn khỏe không?")

# Định dạng nguyên bản của nhà cung cấp (ví dụ: OpenAI)
human_message = HumanMessage(content=[
    {"type": "text", "text": "Xin chào, bạn khỏe không?"},
    {"type": "image_url", "image_url": {"url": "https://example.com/image.jpg"}}
])

# List các content block tiêu chuẩn
human_message = HumanMessage(content_blocks=[
    {"type": "text", "text": "Xin chào, bạn khỏe không?"},
    {"type": "image", "url": "https://example.com/image.jpg"},
])

<div class="alert alert-success">

Việc chỉ định `content_blocks` khi khởi tạo một message vẫn sẽ điền dữ liệu vào `content` của message, nhưng cung cấp một interface an toàn về kiểu dữ liệu để làm việc đó.

</div>

### Các content block tiêu chuẩn

LangChain cung cấp một cách biểu diễn tiêu chuẩn cho nội dung message hoạt động xuyên suốt qua các provider.

Các đối tượng message triển khai một thuộc tính `content_blocks` sẽ parse một cách lazy thuộc tính `content` thành một cấu trúc an toàn và tiêu chuẩn. Ví dụ:

In [5]:
message = AIMessage(
    content=[
        {"type": "thinking", "thinking": "...", "signature": "WaUjzkyp..."},
        {"type": "text", "text": "..."},
    ],
    response_metadata={"model_provider": "anthropic"}
)
message.content_blocks

[{'type': 'reasoning',
  'reasoning': '...',
  'extras': {'signature': 'WaUjzkyp...'}},
 {'type': 'text', 'text': '...'}]

In [4]:
message = AIMessage(
    content=[
        {
            "type": "reasoning",
            "id": "rs_abc123",
            "summary": [
                {"type": "summary_text", "text": "tóm tắt 1"},
                {"type": "summary_text", "text": "tóm tắt 2"},
            ],
        },
        {"type": "text", "text": "...", "id": "msg_abc123"},
    ],
    response_metadata={"model_provider": "openai"}
)
message.content_blocks

[{'type': 'reasoning', 'id': 'rs_abc123', 'reasoning': 'tóm tắt 1'},
 {'type': 'reasoning', 'id': 'rs_abc123', 'reasoning': 'tóm tắt 2'},
 {'type': 'text', 'text': '...', 'id': 'msg_abc123'}]

Hãy xem [hướng dẫn về các tích hợp](/oss/python/integrations/providers/overview) để bắt đầu với provider mà bạn chọn.

<div class="alert alert-info">

**Serialize nội dung tiêu chuẩn**

Nếu một ứng dụng bên ngoài LangChain cần truy cập vào biểu diễn tiêu chuẩn của content block, bạn có thể chọn lưu trữ các content block bên trong nội dung message.

Để làm điều này, bạn có thể đặt biến môi trường `LC_OUTPUT_VERSION` thành `v1`. Hoặc, khởi tạo bất kỳ chat model nào với `output_version="v1"`:

```python
model = init_chat_model("gpt-5-nano", output_version="v1")
```

</div>

### Multimodal

**Đa phương thức** đề cập đến khả năng làm việc với dữ liệu có nhiều định dạng khác nhau, chẳng hạn như văn bản, âm thanh, hình ảnh và video. LangChain bao gồm các kiểu tiêu chuẩn cho những dữ liệu này để có thể sử dụng chéo giữa các nhà cung cấp.

[Chat model](https://docs.langchain.com/oss/python/langchain/models) có thể chấp nhận dữ liệu multimodal làm đầu vào và tạo nó ra làm đầu ra. Dưới đây chúng tôi hiển thị ví dụ ngắn gọn về các message đầu vào có tính năng nhận dữ liệu multimodal.

<Note>

Các key bổ sung có thể được bao gồm ở cấp độ cao nhất trong content block hoặc lồng trong `"extras": {"key": value}`.

Ví dụ, [OpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai) yêu cầu tên tệp cho tài liệu PDF. Hãy xem [trang thông tin provider](https://docs.langchain.com/oss/python/integrations/providers/overview) tương ứng với model bạn chọn để biết thêm chi tiết.

</Note>

**Đầu vào hình ảnh**

In [6]:
# Từ URL
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Mô tả nội dung của hình ảnh này."},
        {"type": "image", "url": "https://example.com/path/to/image.jpg"},
    ]
}

# Từ dữ liệu base64
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Mô tả nội dung của hình ảnh này."},
        {
            "type": "image",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "image/jpeg",
        },
    ]
}

# Từ File ID do nhà cung cấp quản lý
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Mô tả nội dung của hình ảnh này."},
        {"type": "image", "file_id": "file-abc123"},
    ]
}

**Đầu vào tài liệu PDF**

In [ ]:
# Từ URL
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Mô tả nội dung của tài liệu này."},
        {"type": "file", "url": "https://example.com/path/to/document.pdf"},
    ]
}

# Từ dữ liệu base64
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Mô tả nội dung của tài liệu này."},
        {
            "type": "file",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "application/pdf",
        },
    ]
}

# Từ File ID do nhà cung cấp quản lý
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Mô tả nội dung của tài liệu này."},
        {"type": "file", "file_id": "file-abc123"},
    ]
}

**Đầu vào audio**

In [ ]:
# Từ dữ liệu base64
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Mô tả nội dung của audio này."},
        {
            "type": "audio",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "audio/wav",
        },
    ]
}

# Từ File ID do nhà cung cấp quản lý
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Mô tả nội dung của audio này."},
        {"type": "audio", "file_id": "file-abc123"},
    ]
}

**Đầu vào video**

In [ ]:
# Từ dữ liệu base64
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Mô tả nội dung của video này."},
        {
            "type": "video",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "video/mp4",
        },
    ]
}

# Từ File ID do nhà cung cấp quản lý
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Mô tả nội dung của video này."},
        {"type": "video", "file_id": "file-abc123"},
    ]
}

<div class="alert alert-warning">

Không phải tất cả model đều hỗ trợ mọi loại tệp. Hãy kiểm tra [tài liệu tham khảo](https://reference.langchain.com/python/integrations/?_gl=1*3pv23u*_gcl_au*MjA5MzEyMzM1NS4xNzg3MTg5OTgx*_ga*MTg4Njg5NDgwMS4xNzcwMzYxMzE1*_ga_47WX3HKKY2*czE3ODc4ODQ2NjckbzY1JGcxJHQxNzg3ODg1MTI0JGo0NyRsMCRoMA..) của model provider để biết các định dạng được hỗ trợ và giới hạn kích thước.

</div>

### Tham chiếu content block

Các content block được thể hiện (khi tạo một message hoặc truy cập thuộc tính `content_blocks`) dưới dạng một list các dictionary có định kiểu. Mỗi mục trong list phải tuân thủ một trong các loại block sau:

1\. Core

1.1. TextContentBlock

**Mục đích:** Đầu ra văn bản tiêu chuẩn

- **type** (`string`, required): Luôn là `"text"`
- **text** (`string`, required): Nội dung văn bản
- **annotations** (`object[]`): List các chú thích cho văn bản
- **extras** (`object`): Dữ liệu bổ sung đặc thù của nhà cung cấp

**Ví dụ:**

```python
{
    "type": "text",
    "text": "Chào thế giới",
    "annotations": []
}
```

1.2. ReasoningContentBlock

**Mục đích:** Các bước suy luận của model

- **type** (`string`, required): Luôn là `"reasoning"`
- **reasoning** (`string`): Nội dung suy luận
- **extras** (`object`): Dữ liệu bổ sung đặc thù của nhà cung cấp

**Ví dụ:**

```python
{
    "type": "reasoning",
    "reasoning": "Người dùng đang hỏi về...",
    "extras": {"signature": "abc123"},
}
```

---

2\. Multimodal

2.1. ImageContentBlock

**Mục đích:** Dữ liệu hình ảnh

- **type** (`string`, required): Luôn là `"image"`
- **url** (`string`): URL trỏ đến vị trí hình ảnh.
- **base64** (`string`): Dữ liệu hình ảnh được mã hóa Base64.
- **id** (`string`): Định danh duy nhất cho content block này (do nhà cung cấp hoặc LangChain tạo).
- **mime_type** (`string`): [MIME type](https://www.iana.org/assignments/media-types/media-types.xhtml#image) của hình ảnh (ví dụ: `image/jpeg`, `image/png`). Bắt buộc đối với dữ liệu base64.

2.2. AudioContentBlock

**Mục đích:** Dữ liệu âm thanh

- **type** (`string`, required): Luôn là `"audio"`
- **url** (`string`): URL trỏ đến vị trí âm thanh.
- **base64** (`string`): Dữ liệu âm thanh được mã hóa Base64.
- **id** (`string`): Định danh duy nhất cho content block này (do nhà cung cấp hoặc LangChain tạo).
- **mime_type** (`string`): [MIME type](https://www.iana.org/assignments/media-types/media-types.xhtml#audio) của âm thanh (ví dụ: `audio/mpeg`, `audio/wav`). Bắt buộc đối với dữ liệu base64.

2.3. VideoContentBlock

**Mục đích:** Dữ liệu video

- **type** (`string`, required): Luôn là `"video"`
- **url** (`string`): URL trỏ đến vị trí video.
- **base64** (`string`): Dữ liệu video được mã hóa Base64.
- **id** (`string`): Định danh duy nhất cho content block này (do nhà cung cấp hoặc LangChain tạo).
- **mime_type** (`string`): [MIME type](https://www.iana.org/assignments/media-types/media-types.xhtml#video) của video (ví dụ: `video/mp4`, `video/webm`). Bắt buộc đối với dữ liệu base64.

2.4. FileContentBlock

**Mục đích:** Các tệp chung (PDF, v.v.)

- **type** (`string`, required): Luôn là `"file"`
- **url** (`string`): URL trỏ đến vị trí tệp.
- **base64** (`string`): Dữ liệu tệp được mã hóa Base64.
- **id** (`string`): Định danh duy nhất cho content block này (do nhà cung cấp hoặc LangChain tạo).
- **mime_type** (`string`): [MIME type](https://www.iana.org/assignments/media-types/media-types.xhtml) của tệp (ví dụ: `application/pdf`). Bắt buộc đối với dữ liệu base64.

2.5 PlainTextContentBlock

**Mục đích:** Văn bản tài liệu (`.txt`, `.md`)

- **type** (`string`, required): Luôn là `"text-plain"`
- **text** (`string`): Nội dung văn bản
- **mime_type** (`string`): [MIME type](https://www.iana.org/assignments/media-types/media-types.xhtml) của văn bản (ví dụ: `text/plain`, `text/markdown`)

---

3\. Tool Calling

3.1. ToolCall

**Mục đích:** Gọi hàm

- **type** (`string`, required): Luôn là `"tool_call"`
- **name** (`string`, required): Tên của tool cần gọi
- **args** (`object`, required): Các đối số (arguments) truyền cho tool
- **id** (`string`, required): Định danh duy nhất cho tool call này

**Ví dụ:**

```python
{
    "type": "tool_call",
    "name": "search",
    "args": {"query": "weather"},
    "id": "call_123"
}
```

3.2. ToolCallChunk

**Mục đích:** Các phân đoạn (fragment) của tool call đang streaming

- **type** (`string`, required): Luôn là `"tool_call_chunk"`
- **name** (`string`): Tên của tool đang được gọi
- **args** (`string`): Đối số của tool chưa hoàn chỉnh (có thể là một JSON chưa hoàn thiện)
- **id** (`string`): Định danh của tool call
- **index** (`number | string`): Vị trí của chunk này trong luồng stream

3.3. InvalidToolCall

**Mục đích:** Các lệnh gọi bị lỗi định dạng, nhằm mục đích bắt các lỗi phân tích cú pháp JSON.

- **type** (`string`, required): Luôn là `"invalid_tool_call"`
- **name** (`string`): Tên của tool đã gọi thất bại
- **args** (`object`): Các đối số truyền cho tool
- **error** (`string`): Mô tả về sự cố xảy ra

---

4\. Thực thi tool phía server

4.1. ServerToolCall

**Mục đích:** Tool call được thực thi phía server.

- **type** (`string`, required): Luôn là `"server_tool_call"`
- **id** (`string`, required): Một định danh liên kết với tool call.
- **name** (`string`, required): Tên của tool sẽ được gọi.
- **args** (`string`, required): Đối số của tool chưa hoàn chỉnh (có thể là một JSON chưa hoàn thiện)

4.2. ServerToolCallChunk

**Mục đích:** Các phân đoạn streaming của tool call phía server

- **type** (`string`, required): Luôn là `"server_tool_call_chunk"`
- **id** (`string`): Một định danh liên kết với tool call.
- **name** (`string`): Tên của tool đang được gọi
- **args** (`string`): Đối số của tool chưa hoàn chỉnh (có thể là một JSON chưa hoàn thiện)
- **index** (`number | string`): Vị trí của chunk này trong luồng stream

4.3. ServerToolResult

**Mục đích:** Kết quả tìm kiếm

- **type** (`string`, required): Luôn là `"server_tool_result"`
- **tool_call_id** (`string`, required): Định danh của server tool call tương ứng.
- **id** (`string`): Định danh liên kết với kết quả của server tool.
- **status** (`string`, required): Trạng thái thực thi của tool phía server. `"success"` hoặc `"error"`.
- **output**: Đầu ra của tool đã thực thi.

---

5\. Các block đặc thù của provider

5.1. NonStandardContentBlock

**Mục đích:** Cơ chế xử lý dự phòng (escape hatch) đặc thù của provider

- **type** (`string`, required): Luôn là `"non_standard"`
- **value** (`object`, required): Cấu trúc dữ liệu đặc thù của provider

**Sử dụng:** Dành cho các tính năng thử nghiệm hoặc độc quyền của provider

Các loại nội dung đặc thù khác của provider có thể được tìm thấy trong [tài liệu tham khảo](https://docs.langchain.com/oss/python/integrations/providers/overview) của từng model provider.

<div class="alert alert-success">

Xem các định nghĩa type chuẩn mực trong [tài liệu tham khảo API](https://reference.langchain.com/python/langchain/messages).

</div>

<div class="alert alert-info">

Content block đã được giới thiệu như một thuộc tính mới trên message trong LangChain v1 để chuẩn hóa các định dạng nội dung trên toàn bộ các nhà cung cấp đồng thời vẫn duy trì khả năng tương thích ngược với mã nguồn hiện có.

Content block không phải là sự thay thế cho thuộc tính [`content`](https://reference.langchain.com/python/langchain-core/messages/base/BaseMessage), mà thay vào đó là một thuộc tính mới có thể được sử dụng để truy cập nội dung của một message theo định dạng chuẩn hóa.

</div>

## Serialization

Bạn có thể serialize (tuần tự hóa) các message thành các đối tượng đơn giản để lưu trữ và deserialize (giải tuần tự hóa) chúng trở lại thành các instance message. Điều này hữu ích cho việc duy trì lịch sử cuộc hội thoại và khôi phục các phiên làm việc.

In [ ]:
from langchain_core.load import dumpd, load

message = HumanMessage("Thủ đô của nước Pháp là gì?")

# Serialize thành một dict đơn giản
serialized = dumpd(message)

# Deserialize trở lại thành một đối tượng message
restored = load(serialized)

<div class="alert alert-warning">

**`load()` sẽ khởi tạo các đối tượng Python và có thể gây ra các tác dụng phụ trong quá trình deserialize. Không bao giờ gọi `load()` đối với dữ liệu từ nguồn không đáng tin cậy hoặc chưa được xác thực.**

</div>

## Sử dụng với chat model

[Chat model](https://docs.langchain.com/oss/python/langchain/models) nhận một chuỗi các đối tượng message làm đầu vào và trả về một [`AIMessage`](https://reference.langchain.com/python/langchain-core/messages/ai/AIMessage) làm đầu ra. Các tương tác thường không lưu trạng thái, do đó một vòng lặp hội thoại đơn giản sẽ bao gồm việc gọi một model đi kèm với danh sách các message ngày càng tăng.

Tham khảo các hướng dẫn dưới đây để tìm hiểu thêm:

* Các tính năng có sẵn dùng để [lưu trữ và quản lý lịch sử hội thoại](https://docs.langchain.com/oss/python/langchain/short-term-memory)
* Các chiến lược quản lý cửa sổ ngữ cảnh, bao gồm [cắt giảm và tóm tắt các message](https://docs.langchain.com/oss/python/langchain/short-term-memory#common-patterns)